# RoomBeacon — Data Processing & Validation

Một snapshot / một lần Run All. Đọc chart trước kết luận; không sửa source systems.

- [08 — General Data Standardization](#section-08)
- [09 — Address Standardization & Parsing](#section-09)
- [10 — Ward Normalization & Mapping](#section-10)
- [11 — Price / Area Validation](#section-11)
- [12 — Duplicate Analysis](#section-12)
- [13 — Outlier Analysis](#section-13)
- [14 — Cross-field Consistency](#section-14)
- [15 — Analytical Feature Engineering](#section-15)
- [16 — Clean Analytical Dataset](#section-16)

Lý thuyết: [docs](docs/README.md).


In [1]:
# Imports & Configuration — initialize paths, reusable helpers and display settings.
from pathlib import Path
import sys

# Support kernel startup from either the repository root or notebooks/.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'crawler').is_dir() and (p / 'analytics').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))
from notebooks.utils import setup_project_path
PROJECT_ROOT = setup_project_path()

import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown
from dotenv import load_dotenv
from analytics.duckdb.connection import create_analytics_connection
from notebooks.utils.notebook_audit import (
    load_snapshot, ADDRESS_FIELDS,
)
from notebooks.utils.eda_visualization import counts, bar, heatmap, finding

from notebooks.utils.text_standardization import (
    apply_text_standardization, normalize_unicode, normalize_whitespace,
)
from notebooks.utils.address_parser import apply_address_parsing
from notebooks.utils.ward_normalization import apply_ward_mapping
from notebooks.utils.notebook_audit import validate_numeric_candidates
from notebooks.utils.location_analysis import (
    add_spatial_search_mode, audit_coordinate_trust, derive_local_price_bands,
    filter_listings_by_price, get_listings_within_radius,
    summarize_local_price_by_radius, validate_reference_location,
)


load_dotenv(PROJECT_ROOT / '.env', override=False)
pd.set_option('display.max_rows', 16)
pd.set_option('display.max_columns', 12)
px.defaults.template = 'plotly_white'
pio.renderers.default = 'plotly_mimetype+notebook_connected'
SEED = 42

Đã phát hiện thư mục gốc của dự án RoomBeacon: /data/projects/roombeacon/source/roombeacon-system


## Snapshot contract
Grain: một row / `rental_post_id`. Chỉ đọc `v_latest_posts` một lần; raw price/area lấy từ child row có ID mới nhất của contributing version. Join dùng **post ID + observation time**, tie-break bằng version ID như canonical view; không join timestamp độc lập. Address có thể được kế thừa hoặc enrich nên không coi là raw của cùng event.

Một SQL statement materialize listing + evidence. Các upstream scans không phải cam kết snapshot transaction toàn MySQL; alignment flags chặn recovery khi phát hiện khác biệt. Mỗi notebook có RUN_CONTEXT riêng, không so sánh số tuyệt đối giữa hai lần chạy như cùng snapshot. Không lưu secrets trong output.

In [2]:
# Database Connection — canonical factory; one private analytical catalog per kernel.
conn = create_analytics_connection(db_path=':memory:')

In [3]:
# Materialize one runtime snapshot and its narrowly selected raw evidence.
df_latest, raw_evidence, RUN_CONTEXT = load_snapshot(conn, PROJECT_ROOT)
source = df_latest.source_code.fillna('(missing source)')
N_ROWS = RUN_CONTEXT['row_count']
display(pd.Series(RUN_CONTEXT, name='runtime value').to_frame())

,runtime value
snapshot_started_utc,2026-09-25T19:45:32.127146+00:00
snapshot_completed_utc,2026-09-25T19:45:46.124525+00:00
row_count,125591
field_count,24
source_count,9
view_name,v_latest_posts
grain,one row per rental_post_id
raw_grain,latest child ID of contributing version; post ...
max_latest_observed_at,2026-09-25 19:25:20


## Location-first analytical configuration

Điền một reference location thực tế để chạy preview. `None` là trạng thái fail-closed mặc định: notebook vẫn audit location và PASS, nhưng không tạo distance hoặc local-market output giả.

In [4]:
# Configure one real user reference before running radius-based analysis.
REFERENCE_LOCATION = None
# Example shape only: {'name': 'User-selected place', 'latitude': ..., 'longitude': ...}
RADIUS_KM = [1, 3, 5, 10]
SEARCH_CONFIG = {
    'reference_location': REFERENCE_LOCATION,
    'radius_km': 3,
    'min_price': None,
    'max_price': None,
    'price_band': None,
    'min_area': None,
    'max_area': None,
}
assert SEARCH_CONFIG['radius_km'] in RADIUS_KM

<a id="section-08"></a>

# 08 — General Data Standardization

**Câu hỏi:** Normalization thay đổi bao nhiêu values và có idempotent không?

In [5]:
# Standardize into new columns; preserve original strings and all identities.
df_work = df_latest.copy()
text_fields = ['title_raw', *ADDRESS_FIELDS]
field_labels = {
    'title_raw': 'Tiêu đề', 'full_address_text': 'Địa chỉ chi tiết',
    'location_raw': 'Vị trí mô tả', 'best_address_text': 'Địa chỉ tốt nhất',
}
rule_labels = {
    'Unicode NFC': 'Chuẩn hóa Unicode', 'Whitespace': 'Chuẩn hóa khoảng trắng',
    'Combined + empty to null': 'Kết hợp và đổi chuỗi rỗng thành NULL',
}
normalization_rows, examples = [], []
for field in text_fields:
    raw = df_latest[field]
    unicode = raw.map(normalize_unicode)
    whitespace = raw.map(normalize_whitespace)
    df_work = apply_text_standardization(df_work, field, f'{field}_normalized')
    normalized = df_work[f'{field}_normalized']
    for rule, values in [('Unicode NFC', unicode), ('Whitespace', whitespace), ('Combined + empty to null', normalized)]:
        changed = ~(raw.eq(values).fillna(False) | (raw.isna() & values.isna()))
        normalization_rows.append({'field': field_labels[field], 'rule': rule_labels[rule], 'count': int(changed.sum())})
    different = ~(raw.eq(normalized).fillna(False) | (raw.isna() & normalized.isna()))
    sample = pd.DataFrame({'field': field_labels[field], 'before': raw[different], 'after': normalized[different]}).head(3)
    examples.append(sample)
    repeated = apply_text_standardization(df_work, f'{field}_normalized', 'again')['again']
    pd.testing.assert_series_equal(normalized, repeated, check_names=False)
normalization_summary = pd.DataFrame(normalization_rows)
assert len(df_work) == N_ROWS
pd.testing.assert_frame_equal(df_work[df_latest.columns], df_latest)
fig = bar(
    normalization_summary, 'field', 'count', 'Số giá trị thay đổi sau từng quy tắc chuẩn hóa',
    color='rule', barmode='group',
    labels={'field': 'Trường dữ liệu', 'count': 'Số giá trị thay đổi', 'rule': 'Quy tắc'},
)
display(pd.concat(examples).reset_index(drop=True))
finding(f"Combined normalization thay đổi **{normalization_summary.loc[normalization_summary.rule.eq(rule_labels['Combined + empty to null']), 'count'].sum():,}** field values.",
        'Idempotency và bảo toàn source columns đã được assert; số field values không phải số listings duy nhất.')


,field,before,after
0,Tiêu đề,Phòng trọ: Cho Thuê Trọ Ký Túc Xá Khu Chế Xuất...,Phòng trọ: Cho Thuê Trọ Ký Túc Xá Khu Chế Xuất...
1,Tiêu đề,CHO THUÊ TẦNG TRỆT (4x20m) - HẺM 891//Nguyê...,CHO THUÊ TẦNG TRỆT (4x20m) - HẺM 891//Nguyễn K...
2,Tiêu đề,Cho thuê phòng trọ: Cho Thuê Phòng Ngay Đại ...,Cho thuê phòng trọ: Cho Thuê Phòng Ngay Đại Họ...
3,Địa chỉ chi tiết,"Nguyễn Tri Phương, , Vườn Lài TP. Hồ Chí Minh","Nguyễn Tri Phương, , Vườn Lài TP. Hồ Chí Minh"
4,Địa chỉ chi tiết,"223, Bà Hạt, , Vườn Lài TP. Hồ Chí Minh","223, Bà Hạt, , Vườn Lài TP. Hồ Chí Minh"
5,Địa chỉ chi tiết,"93 Gò Dầu, P Tân Quý, Q Tân Phú - Gần Tân Quý,...","93 Gò Dầu, P Tân Quý, Q Tân Phú - Gần Tân Quý,..."
6,Vị trí mô tả,"Vườn Lài, TP. Hồ Chí Minh","Vườn Lài, TP. Hồ Chí Minh"
7,Vị trí mô tả,"Vườn Lài, TP. Hồ Chí Minh","Vườn Lài, TP. Hồ Chí Minh"
8,Vị trí mô tả,"Tân Thuận, TP. Hồ Chí Minh","Tân Thuận, TP. Hồ Chí Minh"
9,Địa chỉ tốt nhất,"Nguyễn Tri Phương, , Vườn Lài TP. Hồ Chí Minh","Nguyễn Tri Phương, , Vườn Lài TP. Hồ Chí Minh"


### Quan sát
Combined normalization thay đổi **2,484** field values.

### Kết luận
Idempotency và bảo toàn source columns đã được assert; số field values không phải số listings duy nhất.

<a id="section-09"></a>

# 09 — Address Standardization & Parsing

**Câu hỏi:** Parser trích xuất thành phần nào, ở nguồn và address layer nào?

In [6]:
# Parse each address layer once; retain best-address outputs for all later sections.
parsed_layers = {}
components = ['street_text_extracted', 'ward_text_extracted', 'district_text_extracted', 'province_text_extracted']
component_labels = {
    'street_text_extracted': 'Đường', 'ward_text_extracted': 'Phường/Xã',
    'district_text_extracted': 'Quận/Huyện', 'province_text_extracted': 'Tỉnh/Thành phố',
}
parse_labels = {
    'PARSED': 'Đã parse', 'PARTIALLY_PARSED': 'Parse một phần', 'AMBIGUOUS': 'Mơ hồ',
    'NO_ADDRESS_EVIDENCE': 'Không có dữ liệu địa chỉ',
    'UNRECOGNIZED_FORMAT': 'Không nhận diện được định dạng',
}
for layer in ADDRESS_FIELDS:
    parsed_layers[layer] = apply_address_parsing(df_work, f'{layer}_normalized')[components + ['parse_status']]
df_work = df_work.join(parsed_layers['best_address_text'])
assert len(df_work) == N_ROWS
parse_status = counts(df_work.parse_status, 'parse_status')
parse_status['status_label'] = parse_status.parse_status.map(parse_labels).fillna(parse_status.parse_status)
parse_status['bar_label'] = parse_status.apply(lambda row: f"{row['count']:,.0f} ({row['percent']:.1f}%)", axis=1)
component_coverage = pd.DataFrame({
    'component': [component_labels[field] for field in components],
    'percent': df_work[components].notna().mean().values * 100,
})
fig = bar(
    parse_status, 'status_label', 'count', 'Kết quả phân tích địa chỉ tốt nhất hiện có',
    orientation='h', text='bar_label',
    labels={'status_label': 'Trạng thái', 'count': 'Số tin đăng'},
)
fig = bar(
    component_coverage, 'component', 'percent', 'Độ phủ của các thành phần địa chỉ',
    order=component_coverage.component.tolist(),
    labels={'component': 'Thành phần', 'percent': 'Độ phủ (%)'},
)


In [7]:
# Compare component coverage within each source and yield across address layers.
component_by_source = df_work[components].notna().groupby(source).mean() * 100
component_by_source = component_by_source.rename(columns=component_labels)
fig = heatmap(
    component_by_source, 'Độ phủ thành phần địa chỉ theo nguồn', percent=True,
    x_title='Thành phần địa chỉ', y_title='Nguồn dữ liệu',
)
layer_labels = {
    'full_address_text': 'Địa chỉ chi tiết',
    'location_raw': 'Vị trí mô tả',
    'best_address_text': 'Địa chỉ tốt nhất hiện có',
}
yield_rows = []
for layer, parsed in parsed_layers.items():
    yield_rows.append({'layer': layer_labels[layer], 'percent': parsed[components].notna().any(axis=1).mean() * 100})
yield_summary = pd.DataFrame(yield_rows)
fig = bar(
    yield_summary, 'layer', 'percent', 'Khả năng trích xuất thành phần từ từng lớp địa chỉ',
    order=yield_summary.layer.tolist(),
    labels={'layer': 'Lớp địa chỉ', 'percent': 'Tỷ lệ trích xuất được (%)'},
)
unrecognized = df_work.parse_status.eq('UNRECOGNIZED_FORMAT')
display(df_work.loc[unrecognized, ['source_code', 'best_address_text_normalized']].head(8))
finding(f"**{unrecognized.sum():,}** UNRECOGNIZED; "
        f"any-component coverage **{df_work[components].notna().any(axis=1).mean():.2%}**.",
        'Không có evidence grouping đã kiểm chứng cho UNRECOGNIZED; sample trên không chứng minh contamination của toàn bộ nhóm.')


,source_code,best_address_text_normalized
71,cafeland,"25/16/1, , Tăng Nhơn Phú TP. Hồ Chí Minh"
85,cafeland,Trung Mỹ Tây TP. Hồ Chí Minh
87,mogi,"306 Gò Dầu, P Tân Quý, Q Tân Phú - Gần Tân Quý..."
92,cafeland,"Nguyễn Tri Phương, , Vườn Lài TP. Hồ Chí Minh"
99,cafeland,"117/97, Nguyễn Hữu Cảnh, , Thạnh Mỹ Tây TP. Hồ..."
107,cafeland,"Nguyễn Trãi, , Cầu Ông Lãnh TP. Hồ Chí Minh"
112,cafeland,"223, Bà Hạt, , Vườn Lài TP. Hồ Chí Minh"
116,cafeland,"60-62, Trần Văn Kỷ, , Bình Thạnh TP. Hồ Chí Minh"


### Quan sát
**35,475** UNRECOGNIZED; any-component coverage **52.01%**.

### Kết luận
Không có evidence grouping đã kiểm chứng cho UNRECOGNIZED; sample trên không chứng minh contamination của toàn bộ nhóm.

<a id="section-10"></a>

# 10 — Ward Normalization & Mapping

**Câu hỏi:** Ward được extract, normalize và resolve đến đâu theo internal reference?

`REFERENCE_STATUS = INTERNAL_ROOMBEACON_REFERENCE`. Mapping dùng reference nội bộ, không phải official external truth. Ambiguous giữ candidates và không ép chọn ward.

In [8]:
# Reuse extracted components and preserve ambiguous mapping candidates.
REFERENCE_STATUS = 'INTERNAL_ROOMBEACON_REFERENCE'
df_work = apply_ward_mapping(df_work, 'ward_text_extracted', 'district_text_extracted')
assert len(df_work) == N_ROWS
ward_status_labels = {
    'MAPPED': 'Đã map', 'UNCHANGED': 'Giữ nguyên', 'UNMAPPED': 'Chưa map được',
    'AMBIGUOUS': 'Mơ hồ', 'MISSING': 'Thiếu dữ liệu', 'NOT_VERIFIED': 'Chưa xác minh',
}
mapping_summary = counts(df_work.ward_mapping_status, 'status')
mapping_summary['status_label'] = mapping_summary.status.map(ward_status_labels).fillna(mapping_summary.status)
mapping_summary['bar_label'] = mapping_summary.apply(lambda row: f"{row['count']:,.0f} ({row['percent']:.1f}%)", axis=1)
fig = bar(
    mapping_summary, 'status_label', 'count', 'Tình trạng chuẩn hóa Phường/Xã',
    orientation='h', text='bar_label',
    labels={'status_label': 'Trạng thái', 'count': 'Số tin đăng'},
)
ward_coverage = pd.DataFrame({'stage': ['Đã trích xuất', 'Đã chuẩn hóa', 'Đã resolve'],
    'percent': [df_work.ward_text_extracted.notna().mean() * 100,
                df_work.ward_normalized.notna().mean() * 100,
                df_work.ward_current.notna().mean() * 100]})
fig = bar(
    ward_coverage, 'stage', 'percent', 'Độ phủ Phường/Xã qua từng bước xử lý',
    order=ward_coverage.stage.tolist(),
    labels={'stage': 'Bước xử lý', 'percent': 'Độ phủ (%)'},
)


In [9]:
# Rank unresolved ward names; absent groups are explicitly reported as empty.
status_titles = {'UNMAPPED': 'Chưa map được', 'AMBIGUOUS': 'Mơ hồ'}
for status in ['UNMAPPED', 'AMBIGUOUS']:
    top = counts(df_work.loc[df_work.ward_mapping_status.eq(status), 'ward_normalized'], 'ward').head(10)
    if top.empty:
        display(Markdown(f'**{status_titles[status]}: 0 trường hợp trong snapshot này.**'))
    else:
        fig = bar(
            top, 'ward', 'count', f"10 tên Phường/Xã {status_titles[status].lower()} xuất hiện nhiều nhất",
            orientation='h', labels={'ward': 'Tên Phường/Xã', 'count': 'Số tin đăng'},
        )
status_by_source = pd.crosstab(source, df_work.ward_mapping_status, normalize='index') * 100
status_by_source = status_by_source.rename(columns=ward_status_labels)
fig = heatmap(
    status_by_source, 'Tình trạng chuẩn hóa Phường/Xã theo nguồn', percent=True,
    x_title='Trạng thái chuẩn hóa', y_title='Nguồn dữ liệu',
)
finding(f"Resolved ward coverage: **{df_work.ward_current.notna().mean():.2%}**; "
        f"ambiguous: **{df_work.ward_mapping_status.eq('AMBIGUOUS').sum():,}**.",
        'ward_current là kết quả internal reference; giữ nguyên extracted/normalized fields để trace.')


### Quan sát
Resolved ward coverage: **29.12%**; ambiguous: **2,798**.

### Kết luận
ward_current là kết quả internal reference; giữ nguyên extracted/normalized fields để trace.

<a id="section-11"></a>

# 11 — Price / Area Validation

**Câu hỏi:** Có thể phục hồi giá/diện tích nào mà không bypass semantic và quality gates?

Reuse Task 11 parsers, semantic classifiers, Decimal scale 2 và domain validators. Price dưới ngưỡng suspicious của Task 11 chỉ bị flag; không filter outlier rows. Chỉ accepted recovery bổ sung physical NULL; existing accepted values luôn được giữ. Parser differences được review, không tự overwrite. Thêm full-string syntax gate để không nhận ranges/dimensions hay price sai đơn vị/period. [Chi tiết contract](docs/11_price_area_validation.md).

In [10]:
# Validate price and area independently against their aligned same-version raw evidence.
price_audit = validate_numeric_candidates(df_latest, raw_evidence, 'price')
area_audit = validate_numeric_candidates(df_latest, raw_evidence, 'area')
df_work['price_amount_clean_candidate'] = price_audit.clean_candidate
df_work['area_value_clean_candidate'] = area_audit.clean_candidate
for kind, audit in [('price', price_audit), ('area', area_audit)]:
    df_work[f'{kind}_validation_action'] = audit.action
    df_work[f'{kind}_regression_status'] = audit.regression
    df_work[f'{kind}_lineage_aligned'] = audit.lineage_aligned
assert len(df_work) == N_ROWS
pd.testing.assert_frame_equal(df_work[df_latest.columns], df_latest)
display(pd.DataFrame({kind: audit.action.value_counts() for kind, audit in [('price', price_audit), ('area', area_audit)]}).fillna(0))

,price,area
action,,
VALIDATED_EXISTING,124763,125088
KEEP_NULL_OR_FLAG,825,490
REPARSE_ACCEPTED_CLEAN,3,13


In [11]:
# Visualize price missing semantics and the sequential recovery quality gates.
audit = price_audit
is_missing = df_latest['price_amount'].isna()
semantic_labels = {
    'NEGOTIABLE_NON_NUMERIC': 'Thỏa thuận / thương lượng',
    'NUMERIC_PARSE_GAP_CANDIDATE': 'Có số nhưng parse chưa được',
    'NO_RAW': 'Không có dữ liệu raw', 'EMPTY_RAW': 'Raw rỗng',
    'WHITESPACE_RAW': 'Raw chỉ có khoảng trắng', 'INVALID_EXISTING': 'Giá trị không hợp lệ',
    'UNCLASSIFIED': 'Chưa phân loại', 'UNKNOWN_LINEAGE': 'Lineage chưa rõ',
}
semantics = counts(audit.loc[is_missing, 'semantic'], 'semantic')
assert semantics['count'].sum() == is_missing.sum()
semantics['semantic_label'] = semantics.semantic.map(semantic_labels).fillna(semantics.semantic)
fig = bar(
    semantics, 'semantic_label', 'count', 'Các dạng thiếu giá thuê', orientation='h',
    labels={'semantic_label': 'Trạng thái', 'count': 'Số tin đăng'},
)
aligned_missing = is_missing & audit.lineage_aligned
parsed_missing = aligned_missing & audit.reparsed.notna()
accepted_domain = parsed_missing & audit.reparse_gate.eq('ACCEPTED_CLEAN')
accepted_recovery = audit.action.eq('REPARSE_ACCEPTED_CLEAN')
stages = ['Đang thiếu giá', 'Lineage khớp', 'Parse được số', 'Đạt kiểm tra miền', 'Phục hồi sạch']
recovery = pd.DataFrame({'stage': stages,
    'count': [is_missing.sum(), aligned_missing.sum(), parsed_missing.sum(), accepted_domain.sum(), accepted_recovery.sum()]})
assert recovery['count'].diff().dropna().le(0).all()
fig = bar(
    recovery, 'stage', 'count', 'Số tin đăng còn lại qua từng bước phục hồi giá',
    order=stages, labels={'stage': 'Bước xử lý', 'count': 'Số tin đăng'},
)


In [12]:
# Audit price regression without assuming differences mean the old parser was wrong.
audit = price_audit
existing_mask = df_latest['price_amount'].notna()
regression_labels = {
    'CANONICAL_MATCH': 'Khớp', 'DIFFERENT_REVIEW': 'Khác — cần xem lại',
    'REPARSE_NULL': 'Parse lại thành NULL', 'UNKNOWN_LINEAGE': 'Lineage chưa rõ',
    'NO_EXISTING': 'Không có giá hiện tại',
}
regression = counts(audit.loc[existing_mask, 'regression'], 'classification')
regression['classification_label'] = regression.classification.map(regression_labels).fillna(regression.classification)
fig = bar(
    regression, 'classification_label', 'count', 'So sánh giá hiện tại và giá parse lại',
    labels={'classification_label': 'Kết quả so sánh', 'count': 'Số tin đăng'},
)
problem_mask = audit.action.eq('KEEP_NULL_OR_FLAG') | audit.regression.isin(['DIFFERENT_REVIEW', 'REPARSE_NULL'])
patterns = audit.loc[problem_mask, 'raw'].astype('string').str.lower().str.replace(r'\d+', '#', regex=True)
pattern_counts = counts(patterns, 'raw_pattern').head(10)
display(Markdown('**10 mẫu biểu diễn giá cần xem lại nhiều nhất**'))
display(pattern_counts)
display(audit.loc[problem_mask, ['raw', 'existing', 'reparsed', 'reparse_gate', 'regression']].head(6))
finding(f"Accepted recovery: **{audit.action.eq('REPARSE_ACCEPTED_CLEAN').sum():,}**; "
        f"differences needing review: **{audit.regression.eq('DIFFERENT_REVIEW').sum():,}**.",
        'Chỉ validated existing + accepted deterministic recovery vào candidate. Không fabricate số cho negotiable; '
        'clean candidate chưa phải Clean Analytical Dataset.')


**10 mẫu biểu diễn giá cần xem lại nhiều nhất**

,raw_pattern,count,percent
0,# triệu # nghìn,7821,87.356193
1,thỏa thuận,564,6.299564
2,#.# triệu/tháng,257,2.870546
3,#.# đồng/tháng,112,1.250977
4,# đồng/tháng,101,1.128113
5,#.#đ,25,0.279236
6,giá từ: #.# đến #.# triệu,23,0.256897
7,# tỷ # triệu,16,0.178711
8,từ liên hệ lấy giá,11,0.122864
9,giá từ: #.# đến # triệu,6,0.067017


,raw,existing,reparsed,reparse_gate,regression
0,3 triệu 100 nghìn,3000000.00,3100000.00,ACCEPTED_CLEAN,DIFFERENT_REVIEW
1,3 triệu 300 nghìn,3000000.00,3300000.00,ACCEPTED_CLEAN,DIFFERENT_REVIEW
3,3 triệu 800 nghìn,3000000.00,3800000.00,ACCEPTED_CLEAN,DIFFERENT_REVIEW
4,3 triệu 700 nghìn,3000000.00,3700000.00,ACCEPTED_CLEAN,DIFFERENT_REVIEW
5,3 triệu 300 nghìn,3000000.00,3300000.00,ACCEPTED_CLEAN,DIFFERENT_REVIEW
6,3 triệu 500 nghìn,3000000.00,3500000.00,ACCEPTED_CLEAN,DIFFERENT_REVIEW


### Quan sát
Accepted recovery: **3**; differences needing review: **8,129**.

### Kết luận
Chỉ validated existing + accepted deterministic recovery vào candidate. Không fabricate số cho negotiable; clean candidate chưa phải Clean Analytical Dataset.

In [13]:
# Visualize area missing semantics and the sequential recovery quality gates.
audit = area_audit
is_missing = df_latest['area_value'].isna()
area_semantic_labels = {
    'NUMERIC_PARSE_GAP_CANDIDATE': 'Có số nhưng parse chưa được',
    'NO_RAW': 'Không có dữ liệu raw', 'EMPTY_RAW': 'Raw rỗng',
    'WHITESPACE_RAW': 'Raw chỉ có khoảng trắng', 'INVALID_EXISTING': 'Giá trị không hợp lệ',
    'RAW_PRESENT_NON_NUMERIC': 'Raw không phải số',
    'UNCLASSIFIED': 'Chưa phân loại', 'UNKNOWN_LINEAGE': 'Lineage chưa rõ',
}
semantics = counts(audit.loc[is_missing, 'semantic'], 'semantic')
assert semantics['count'].sum() == is_missing.sum()
semantics['semantic_label'] = semantics.semantic.map(area_semantic_labels).fillna(semantics.semantic)
fig = bar(
    semantics, 'semantic_label', 'count', 'Các dạng thiếu diện tích', orientation='h',
    labels={'semantic_label': 'Trạng thái', 'count': 'Số tin đăng'},
)
aligned_missing = is_missing & audit.lineage_aligned
parsed_missing = aligned_missing & audit.reparsed.notna()
accepted_domain = parsed_missing & audit.reparse_gate.eq('ACCEPTED_CLEAN')
accepted_recovery = audit.action.eq('REPARSE_ACCEPTED_CLEAN')
stages = ['Đang thiếu diện tích', 'Lineage khớp', 'Parse được số', 'Đạt kiểm tra miền', 'Phục hồi sạch']
recovery = pd.DataFrame({'stage': stages,
    'count': [is_missing.sum(), aligned_missing.sum(), parsed_missing.sum(), accepted_domain.sum(), accepted_recovery.sum()]})
assert recovery['count'].diff().dropna().le(0).all()
fig = bar(
    recovery, 'stage', 'count', 'Số tin đăng còn lại qua từng bước phục hồi diện tích',
    order=stages, labels={'stage': 'Bước xử lý', 'count': 'Số tin đăng'},
)


In [14]:
# Audit area regression without assuming differences mean the old parser was wrong.
audit = area_audit
existing_mask = df_latest['area_value'].notna()
regression = counts(audit.loc[existing_mask, 'regression'], 'classification')
regression['classification_label'] = regression.classification.map(regression_labels).fillna(regression.classification)
fig = bar(
    regression, 'classification_label', 'count', 'So sánh diện tích hiện tại và diện tích parse lại',
    labels={'classification_label': 'Kết quả so sánh', 'count': 'Số tin đăng'},
)
problem_mask = audit.action.eq('KEEP_NULL_OR_FLAG') | audit.regression.isin(['DIFFERENT_REVIEW', 'REPARSE_NULL'])
patterns = audit.loc[problem_mask, 'raw'].astype('string').str.lower().str.replace(r'\d+', '#', regex=True)
pattern_counts = counts(patterns, 'raw_pattern').head(10)
display(Markdown('**10 mẫu biểu diễn diện tích cần xem lại nhiều nhất**'))
display(pattern_counts)
display(audit.loc[problem_mask, ['raw', 'existing', 'reparsed', 'reparse_gate', 'regression']].head(6))
finding(f"Accepted recovery: **{audit.action.eq('REPARSE_ACCEPTED_CLEAN').sum():,}**; "
        f"differences needing review: **{audit.regression.eq('DIFFERENT_REVIEW').sum():,}**.",
        'Chỉ validated existing + accepted deterministic recovery vào candidate. Không fabricate số cho negotiable; '
        'clean candidate chưa phải Clean Analytical Dataset.')


**10 mẫu biểu diễn diện tích cần xem lại nhiều nhất**

,raw_pattern,count,percent
0,(missing),446,91.020408
1,#,34,6.938776
2,# m # # pn # wc,3,0.612245
3,liên hệ,2,0.408163
4,# m #,2,0.408163
5,giường tầng ở ghép,1,0.204082
6,#.# m²,1,0.204082
7,#m #,1,0.204082


,raw,existing,reparsed,reparse_gate,regression
41,NaN,None,None,DOMAIN_INVALID,NO_EXISTING
55,NaN,None,None,DOMAIN_INVALID,NO_EXISTING
67,NaN,None,None,DOMAIN_INVALID,NO_EXISTING
75,NaN,None,None,DOMAIN_INVALID,NO_EXISTING
85,Liên hệ,None,None,DOMAIN_INVALID,NO_EXISTING
89,NaN,None,None,DOMAIN_INVALID,NO_EXISTING


### Quan sát
Accepted recovery: **13**; differences needing review: **0**.

### Kết luận
Chỉ validated existing + accepted deterministic recovery vào candidate. Không fabricate số cho negotiable; clean candidate chưa phải Clean Analytical Dataset.

## Analytical Preview — Radius-Based Local Market

**WHERE? → HOW FAR? → HOW MANY? → HOW MUCH? → WHICH LISTINGS REMAIN?**

Canonical filter order: real reference location → trusted coordinates → radius → validated clean price → optional area → optional administrative filters → sort/present. District/ward search is the fallback branch for `ADMIN_ONLY`; it never fabricates exact distance.

In [15]:
# Identify listings eligible for radius-based search without overwriting coordinates.
df_work = audit_coordinate_trust(df_work)
df_work = add_spatial_search_mode(df_work)
search_eligibility = counts(df_work.spatial_search_mode, 'search_mode')
display(search_eligibility)

reference_is_valid = validate_reference_location(SEARCH_CONFIG['reference_location'])
RADIUS_ANALYTICS_STATUS = 'READY' if reference_is_valid else 'SKIPPED_REFERENCE_LOCATION_REQUIRED'
display(Markdown(
    f"**Radius analytics status: {RADIUS_ANALYTICS_STATUS}.** "
    f"Trusted-coordinate denominator: **{df_work.has_trusted_coordinate.sum():,}** listings."
))

,search_mode,count,percent
0,LOCATION_UNAVAILABLE,68057,54.189393
1,ADMIN_ONLY,55278,44.014300
2,RADIUS_ELIGIBLE,2256,1.796307


**Radius analytics status: SKIPPED_REFERENCE_LOCATION_REQUIRED.** Trusted-coordinate denominator: **2,256** listings.

In [16]:
# Compute cumulative radius coverage and local median price only for a real reference.
radius_summary = pd.DataFrame()
radius_band_summary = pd.DataFrame()
local_price_long = pd.DataFrame()
if reference_is_valid:
    radius_summary, radius_band_summary = summarize_local_price_by_radius(
        df_work, SEARCH_CONFIG['reference_location'], RADIUS_KM,
    )
    assert radius_summary.listing_count.diff().fillna(0).ge(0).all()
    radius_summary['radius_label'] = radius_summary.radius_km.map(lambda value: f'{value:g} km')
    fig = bar(
        radius_summary, 'radius_label', 'listing_count', 'Số tin đăng trong từng bán kính tích lũy',
        text='listing_count', order=radius_summary.radius_label.tolist(),
        labels={'radius_label': 'Bán kính', 'listing_count': 'Số tin đăng'},
    )

    local_frames = []
    for radius in RADIUS_KM:
        local = get_listings_within_radius(
            df_work,
            SEARCH_CONFIG['reference_location']['latitude'],
            SEARCH_CONFIG['reference_location']['longitude'],
            radius,
        )
        local['validated_price'] = pd.to_numeric(local.price_amount_clean_candidate, errors='coerce')
        local['radius_km'] = radius
        local['radius_label'] = f'{radius:g} km'
        local_frames.append(local.loc[local.validated_price.notna(), ['radius_km', 'radius_label', 'validated_price']])
    local_price_long = pd.concat(local_frames, ignore_index=True)
    if not local_price_long.empty:
        median_by_radius = (local_price_long.groupby(['radius_km', 'radius_label'], as_index=False)
                            .validated_price.median()
                            .rename(columns={'validated_price': 'median_validated_price'})
                            .sort_values('radius_km'))
        fig = bar(
            median_by_radius, 'radius_label', 'median_validated_price',
            'Giá thuê trung vị theo bán kính tích lũy',
            text='median_validated_price', order=median_by_radius.radius_label.tolist(),
            labels={'radius_label': 'Bán kính', 'median_validated_price': 'Giá thuê trung vị (VND/tháng)'},
        )
    display(radius_summary)
else:
    display(Markdown(
        '**Listings-by-radius and price-by-radius skipped:** set `REFERENCE_LOCATION` to a real '
        'user-selected point. No distance is inferred from address text or administrative centroids.'
    ))


**Listings-by-radius and price-by-radius skipped:** set `REFERENCE_LOCATION` to a real user-selected point. No distance is inferred from address text or administrative centroids.

In [17]:
# Filter listings before calculating the selected local price distribution.
selected_local = pd.DataFrame()
price_filtered = pd.DataFrame()
final_candidates = pd.DataFrame()
local_band_boundaries = None
EXAMPLE_SEARCH_WORKS = False
if reference_is_valid:
    reference = SEARCH_CONFIG['reference_location']
    selected_local = get_listings_within_radius(
        df_work, reference['latitude'], reference['longitude'], SEARCH_CONFIG['radius_km'],
    )
    selected_local['validated_price'] = pd.to_numeric(
        selected_local.price_amount_clean_candidate, errors='coerce'
    )

    # Derive price bands from the local candidate set, not globally.
    local_bands, local_band_boundaries = derive_local_price_bands(selected_local)
    selected_local['local_price_band'] = local_bands
    price_filtered = filter_listings_by_price(
        selected_local,
        min_price=SEARCH_CONFIG['min_price'],
        max_price=SEARCH_CONFIG['max_price'],
    )
    if SEARCH_CONFIG['price_band'] is not None:
        price_filtered = price_filtered.loc[
            price_filtered.local_price_band.eq(SEARCH_CONFIG['price_band'])
        ].copy()

    # Area is optional and is applied only after radius and price.
    final_candidates = price_filtered.copy()
    clean_area = pd.to_numeric(final_candidates.area_value_clean_candidate, errors='coerce')
    if SEARCH_CONFIG['min_area'] is not None:
        final_candidates = final_candidates.loc[clean_area.ge(SEARCH_CONFIG['min_area'])].copy()
        clean_area = pd.to_numeric(final_candidates.area_value_clean_candidate, errors='coerce')
    if SEARCH_CONFIG['max_area'] is not None:
        final_candidates = final_candidates.loc[clean_area.le(SEARCH_CONFIG['max_area'])].copy()
    EXAMPLE_SEARCH_WORKS = True

    flow = pd.Series({
        'Reference location': reference['name'],
        'Radius (km)': SEARCH_CONFIG['radius_km'],
        'Listings inside radius': len(selected_local),
        'Listings after validated price': len(price_filtered),
        'Final candidates after optional area': len(final_candidates),
    }, name='value')
    display(flow.to_frame())
else:
    display(Markdown('**Example search flow awaiting a real `REFERENCE_LOCATION`; no candidates fabricated.**'))

**Example search flow awaiting a real `REFERENCE_LOCATION`; no candidates fabricated.**

In [18]:
# Visualize selected-radius price distribution, local bands and price-distance association.
if reference_is_valid:
    selected_price = selected_local.loc[selected_local.validated_price.notna(), 'validated_price']
    if not selected_price.empty:
        histogram_data = pd.DataFrame({'validated_price': selected_price})
        fig = px.histogram(
            histogram_data, x='validated_price', nbins=30,
            title=f"Phân bố giá phòng trong bán kính {SEARCH_CONFIG['radius_km']} km",
            labels={'validated_price': 'Giá thuê đã kiểm tra (VND/tháng)', 'count': 'Số tin đăng'},
        )
        fig.update_layout(yaxis_title='Số tin đăng', bargap=0.05)
        fig.show()

    if local_band_boundaries is None:
        display(Markdown('**Local price bands skipped:** sample is too small or tertile boundaries are tied.'))
    else:
        band_labels = {'LOCAL_LOW': 'Giá thấp', 'LOCAL_MEDIUM': 'Giá trung bình', 'LOCAL_HIGH': 'Giá cao'}
        selected_band_summary = counts(selected_local.local_price_band.dropna(), 'local_price_band')
        selected_band_summary['band_label'] = selected_band_summary.local_price_band.map(band_labels)
        boundary_text = (f"Giá thấp ≤ {local_band_boundaries['p33']:,.0f}; "
                         f"giá trung bình ≤ {local_band_boundaries['p67']:,.0f}; "
                         f"giá cao > {local_band_boundaries['p67']:,.0f} VND")
        fig = bar(
            selected_band_summary, 'band_label', 'count',
            f"Nhóm giá trong bán kính {SEARCH_CONFIG['radius_km']} km — {boundary_text}",
            order=['Giá thấp', 'Giá trung bình', 'Giá cao'],
            labels={'band_label': 'Nhóm giá', 'count': 'Số tin đăng'},
        )

    if not radius_band_summary.empty:
        band_totals = radius_band_summary.groupby('radius_km')['count'].transform('sum')
        radius_band_summary['percent'] = radius_band_summary['count'] / band_totals * 100
        radius_band_summary['radius_label'] = radius_band_summary.radius_km.map(lambda value: f'{value:g} km')
        radius_band_summary['band_label'] = radius_band_summary.local_price_band.map(band_labels)
        fig = px.bar(
            radius_band_summary, x='radius_label', y='percent', color='band_label',
            barmode='stack', title='Cơ cấu nhóm giá theo bán kính tích lũy',
            hover_data=['count', 'p33', 'p67'],
            category_orders={'radius_label': [f'{value:g} km' for value in RADIUS_KM],
                             'band_label': ['Giá thấp', 'Giá trung bình', 'Giá cao']},
            labels={'radius_label': 'Bán kính', 'percent': 'Tỷ lệ trong bán kính (%)', 'band_label': 'Nhóm giá'},
        )
        fig.show()

    scatter_data = selected_local.loc[selected_local.validated_price.notna()].copy()
    if not scatter_data.empty:
        fig = px.scatter(
            scatter_data, x='distance_km', y='validated_price',
            hover_data={
                'source_code': True, 'rental_post_id': True, 'district_text_extracted': True,
                'ward_current': True, 'area_value_clean_candidate': True,
                'distance_km': ':.2f', 'validated_price': ':,.0f',
            },
            title='Giá thuê theo khoảng cách tới điểm tham chiếu',
            labels={'distance_km': 'Khoảng cách (km)', 'validated_price': 'Giá thuê đã kiểm tra (VND/tháng)'},
        )
        fig.show()


In [19]:
# Present a compact analytical candidate table sorted by distance, not recommendation rank.
if reference_is_valid:
    candidate_columns = [
        'rental_post_id', 'title_raw', 'distance_km', 'validated_price',
        'area_value_clean_candidate', 'district_text_extracted', 'ward_current',
        'best_address_text', 'source_code',
    ]
    candidate_table = final_candidates[candidate_columns].rename(columns={
        'title_raw': 'title', 'validated_price': 'price',
        'area_value_clean_candidate': 'area', 'district_text_extracted': 'district',
        'ward_current': 'ward', 'best_address_text': 'best_address', 'source_code': 'source',
    }).sort_values(['distance_km', 'price'], na_position='last')
    display(candidate_table.head(20))
    display(Markdown(
        f"Reference **{SEARCH_CONFIG['reference_location']['name']}** → radius "
        f"**{SEARCH_CONFIG['radius_km']} km** → **{len(selected_local):,}** spatial listings → "
        f"**{len(price_filtered):,}** price-filtered → **{len(final_candidates):,}** final candidates."
    ))

<a id="section-12"></a>

# 12 — Duplicate Analysis

> Chưa thực thi trong snapshot notebook hiện tại. Không tạo results, filter rows hoặc training ở bước này.

<a id="section-13"></a>

# 13 — Outlier Analysis

> Chưa thực thi trong snapshot notebook hiện tại. Không tạo results, filter rows hoặc training ở bước này.

<a id="section-14"></a>

# 14 — Cross-field Consistency

> Chưa thực thi trong snapshot notebook hiện tại. Không tạo results, filter rows hoặc training ở bước này.

<a id="section-15"></a>

# 15 — Analytical Feature Engineering

> Chưa thực thi trong snapshot notebook hiện tại. Không tạo results, filter rows hoặc training ở bước này.

<a id="section-16"></a>

# 16 — Clean Analytical Dataset

> Chưa thực thi trong snapshot notebook hiện tại. Không tạo results, filter rows hoặc training ở bước này.

In [20]:
# Assert row/identity/source conservation before closing the shared connection.
assert len(df_work) == N_ROWS == df_work.rental_post_id.nunique()
assert df_work.rental_post_id.equals(df_latest.rental_post_id)
pd.testing.assert_frame_equal(df_work[df_latest.columns], df_latest)
assert df_work.loc[df_work.ward_mapping_status.eq('AMBIGUOUS'), 'ward_current'].isna().all()
assert df_work.loc[df_work.spatial_search_mode.eq('RADIUS_ELIGIBLE'), 'has_trusted_coordinate'].all()
assert not (df_work.spatial_search_mode.eq('LOCATION_UNAVAILABLE') & df_work.has_trusted_coordinate).any()
conn.close()
display(Markdown('**Boundary assertions PASS.** Sections 08–11 executed; 12–16 remain pending. No modeling.'))

**Boundary assertions PASS.** Sections 08–11 executed; 12–16 remain pending. No modeling.